In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten, Reshape, Input, Layer, BatchNormalization, Dropout


(x_train, y_train), (x_test, y_test) = mnist.load_data()

# стандартизация входных данных
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test  = np.reshape(x_test,  (len(x_test),  28, 28, 1))


hidden_dim = 2
batch_size = 60 # должно быть кратно 60 000

def dropout_and_batch(x):
    return Dropout(0.3)(BatchNormalization()(x))

input_img = Input((28, 28, 1))
x = Flatten()(input_img)
x = Dense(256, activation='relu')(x)
x = dropout_and_batch(x)
x = Dense(128, activation='relu')(x)
x = dropout_and_batch(x)

z_mean = Dense(hidden_dim)(x)
z_log_var = Dense(hidden_dim)(x)

def noiser(args):
    mean, log_var = args
    # batch ölçüsü sabit yazılmır, dinamik götürülür
    N = tf.random.normal(shape=tf.shape(mean), mean=0., stddev=1.0)
    return tf.exp(log_var / 2) * N + mean

class Sampler(Layer):
    # Keras 3-də KerasTensor-ları tf funksiyalarına birbaşa vermək olmur,
    # ona görə sampling və KL loss layer daxilində hesablanır
    def call(self, inputs):
        mean, log_var = inputs
        kl_loss = -0.5 * tf.reduce_sum(1 + log_var - tf.square(mean) - tf.exp(log_var), axis=-1)
        self.add_loss(tf.reduce_mean(kl_loss))
        return noiser(inputs)

    def compute_output_shape(self, input_shape):
        return input_shape[0]

h = Sampler()([z_mean, z_log_var])

input_dec = Input(shape=(hidden_dim,))
d = Dense(128, activation='relu')(input_dec)
d = dropout_and_batch(d)
d = Dense(256, activation='relu')(d)
d = dropout_and_batch(d)
d = Dense(28*28, activation='sigmoid')(d)
decoded = Reshape((28, 28, 1))(d)

encoder = keras.Model(input_img, h, name='encoder')
decoder = keras.Model(input_dec, decoded, name='decoder')
vae = keras.Model(input_img, decoder(encoder(input_img)), name="vae")


def vae_loss(x, y):
    x = tf.reshape(x, (-1, 28*28))
    y = tf.reshape(y, (-1, 28*28))
    loss = tf.reduce_sum(tf.square(x - y), axis=-1)
    return loss  # KL loss Sampler layer-in içində əlavə olunur

vae.compile(optimizer='adam', loss=vae_loss)

vae.fit(x_train, x_train, epochs=5, batch_size=batch_size, shuffle=True)


h = encoder.predict(x_test[:6000], batch_size=batch_size)
plt.scatter(h[:, 0], h[:, 1], s=2)
plt.show()


n = 5
total = 2*n+1

plt.figure(figsize=(total/2, total/2))

num = 1
for i in range(-n, n+1):
    for j in range(-n, n+1):
        ax = plt.subplot(total, total, num)
        num += 1
        img = decoder.predict(np.expand_dims([3*i/n, 3*j/n], axis=0), verbose=0)
        plt.imshow(img.squeeze(), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)

plt.show()

## Izah

In [ ]:
import numpy as np
# NumPy — massivlərlə işləmək üçün.
# Bu kodda:
# - np.reshape      → massivin formasını dəyişmək (28x28 → 28x28x1)
# - np.expand_dims  → massivə əlavə ox (ölçü) əlavə etmək: (2,) → (1, 2)


import matplotlib.pyplot as plt
# matplotlib — şəkilləri və qrafikləri göstərmək üçün.
# Bu kodda:
# - plt.scatter  → gizli fəzanın nöqtə buludu
# - plt.subplot  → bir figur daxilində şəbəkə (grid) şəklində kiçik qrafiklər
# - plt.imshow   → massivi şəkil kimi göstərmək


import tensorflow as tf
# TensorFlow-un özü. Əvvəlki kodlarda yox idi, bu kodda LAZIMDIR, çünki
# aşağıda aşağı səviyyəli əməliyyatlardan istifadə olunur:
#   tf.random.normal  → normal paylanmadan təsadüfi ədədlər (səs-küy)
#   tf.exp, tf.square → eksponent, kvadrat
#   tf.reduce_sum / tf.reduce_mean → cəmləmə / ortalama
#   tf.reshape, tf.shape → forma əməliyyatları


from tensorflow.keras.datasets import mnist
# MNIST — əl ilə yazılmış rəqəmlərin (0-dan 9-a qədər) data bazası.
#   - 60000 öyrətmə şəkli, 10000 test şəkli
#   - hər şəkil 28x28 piksel, boz tonlu, piksel qiyməti 0..255


from tensorflow import keras
# Keras-ın özü. Bu kodda keras.Model(...) üçün lazımdır.


from tensorflow.keras.layers import Dense, Flatten, Reshape, Input, Layer, BatchNormalization, Dropout
# Dense              → tam bağlı qat
# Flatten            → (28, 28, 1) → (784,)
# Reshape            → (784,) → (28, 28, 1)
# Input              → modelin giriş formasını elan edir
# Layer              → öz qatımızı yazmaq üçün baza sinif (aşağıda Sampler)
# BatchNormalization → hər batch daxilində aktivasiyaları normallaşdırır
#                      (orta ≈ 0, dispersiya ≈ 1). Öyrənməni sabitləşdirir.
# Dropout            → öyrətmə zamanı neyronların bir hissəsini təsadüfi
#                      "söndürür" (overfitting-ə qarşı)


# =====================================================================
# BU KOD NƏDİR? — VAE (Variational Autoencoder / Variasiyalı avtoenkoder)
# =====================================================================
#
# Əvvəlki kodla (2 ölçülü adi avtoenkoder) müqayisədə baza memarlıq eynidir:
# şəkil → 2 ədəd → şəkil. FƏRQ darboğazın NƏ OLDUĞUNDADIR.
#
#   ADİ AVTOENKODER:  şəkil → TƏK bir nöqtə (x, y) → şəkil
#   VAE:              şəkil → bir PAYLANMA (orta + yayılma) → həmin
#                     paylanmadan TƏSADÜFİ nöqtə seçilir → şəkil
#
# Yəni encoder hər şəkil üçün "bu şəkil gizli fəzada təxminən BURADADIR
# (mean), və bu qədər qeyri-müəyyənliklə (log_var)" deyir. Sonra bu
# paylanmadan təsadüfi bir nöqtə götürülür və decoder ona baxıb şəkil çəkir.
#
#   şəkil ──► [ENCODER] ──► mean, log_var ──► [SAMPLER] ──► z (2 ədəd) ──► [DECODER] ──► şəkil
#                                                │
#                                                └─► KL itkisi (əlavə cərimə)
#
# NİYƏ BELƏ MURƏKKƏB? Adi avtoenkoderdə gizli fəza "dəlikli" olur: nöqtə
# buludunun arasında boşluqlar var və orada decoder mənasız şey çəkir
# (əvvəlki kodda [50, 250] problemi məhz bu idi). VAE iki mexanizmlə bunu həll edir:
#   1) Səs-küy (noiser): eyni şəklin kodu hər dəfə azca fərqli çıxır, ona görə
#      decoder ətraf bölgəni də "öyrənməyə" məcbur olur → boşluqlar azalır.
#   2) KL itkisi: bütün paylanmaları standart normal N(0, 1) ətrafına
#      toplayır → nöqtələr 0 ətrafında, təxminən -3..3 aralığında yığılır.
#
# NƏTİCƏ: gizli fəzadan istənilən nöqtəni (0-a yaxın) seçsək, decoder
# MƏNALI rəqəm çəkir. Yəni model həqiqi GENERATORdur.
#
# MÜQAYİSƏ:
#
#                        Adi 2D avtoenkoder        Bu kod (VAE)
# Darboğaz çıxışı:       1 nöqtə (x, y)            mean + log_var → seçilmiş z
# Gizli fəzanın miqyası: nəzarətsiz (0..300 ola    0 ətrafında, ~(-3..3)
#                        bilər)
# Loss:                  yalnız MSE                bərpa xətası + KL
# Normallaşdırma/Dropout: yoxdur                   var (BatchNorm + Dropout)
# epochs / batch_size:   10 / 64                   5 / 60
# Yeni şəkil yaratmaq:   riskli (nöqtə kənarda     etibarlı (nöqtəni -3..3
#                        ola bilər)                aralığından seçmək kifayətdir)


# ---------------------------------------------------------------------
# 1) DATANIN YÜKLƏNMƏSİ VƏ HAZIRLANMASI
# ---------------------------------------------------------------------

(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Dörd massiv qaytarır:
#   x_train → (60000, 28, 28), uint8 (0..255)
#   y_train → (60000,) rəqəmlər (bu kodda İSTİFADƏ OLUNMUR)
#   x_test  → (10000, 28, 28)
#   y_test  → (10000,)  (bu kodda da istifadə olunmur)

# стандартизация входных данных
x_train = x_train.astype('float32') / 255
x_test = x_test.astype('float32') / 255
# (Rusca şərh: "giriş datasının standartlaşdırılması".)
# Piksellər 0..255 → 0..1. Səbəb: neyron şəbəkələr kiçik ədədlərlə yaxşı
# öyrənir və decoder-in sigmoid çıxışı da 0..1 aralığındadır.
# Əvvəlki koddan FƏRQ: burada .astype('float32') var. Əvvəl bölmədən
# sonra float64 alınırdı (Keras sonra özü float32-yə çevirirdi). İndi
# ilk başdan float32 olur: yaddaş 2 dəfə az, TensorFlow-un öz tipi ilə uyğun,
# və itki hesablamasında tip uyğunsuzluğu xətası çıxmır.

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test  = np.reshape(x_test,  (len(x_test),  28, 28, 1))
# Kanal ölçüsü əlavə edir:
#   (60000, 28, 28) → (60000, 28, 28, 1)
#   (10000, 28, 28) → (10000, 28, 28, 1)
# Son 1 → tək kanal (boz ton). Elementlərin sayı dəyişmir.


# ---------------------------------------------------------------------
# 2) HİPERPARAMETRLƏR VƏ KÖMƏKÇİ FUNKSİYA
# ---------------------------------------------------------------------

hidden_dim = 2
# Gizli fəzanın ölçüsü: hər şəkil 2 ədədlə təsvir olunur.
# 2 seçilməsinin səbəbi əvvəlki kodla eynidir: müstəvidə görmək üçün.
# (Real tətbiqlərdə adətən 16, 32, 64 və s. seçilir; keyfiyyət artır,
# amma görüntü itir.)

batch_size = 60 # должно быть кратно 60 000
# Rusca şərh: "60 000-ə bölünməlidir". 60000 / 60 = 1000 → hər epoch-da
# DƏQİQ 1000 addım, yarımçıq son batch yoxdur.
# QEYD: bu kod dinamik forma işlədir (tf.shape(mean)), ona görə bu şərt
# əslində MƏCBURİ deyil. Bu şərh köhnə yazılışdan qalıb, orada batch ölçüsü
# sabit idi. Yenə də 60-ın seçilməsi zərər vermir.
# 6000 (aşağıda scatter üçün) də 60-a bölünür → predict-də də yarımçıq batch yoxdur.

def dropout_and_batch(x):
    return Dropout(0.3)(BatchNormalization()(x))
# Köməkçi funksiya: giriş x-i əvvəl BatchNormalization-dan, sonra Dropout-dan
# keçirir. İçəridən kənara oxuyun: BatchNormalization()(x) → Dropout(0.3)(...).
#
# BatchNormalization:
#   - hər batch üçün aktivasiyaları orta=0, dispersiya=1 olacaq şəkildə
#     şkalalayır, sonra öyrənilən 2 parametrlə (gamma, beta) yenidən miqyaslayır.
#   - öyrənməni sürətləndirir və sabitləşdirir.
#   - Parametr: hər neyron üçün 4 ədəd (gamma, beta öyrənilir; hərəkətli orta
#     və hərəkətli dispersiya öyrənilmir, yalnız yenilənir).
#
# Dropout(0.3):
#   - ÖYRƏTMƏ zamanı neyronların 30%-ni təsadüfi 0 edir (qalanları 1/0.7 dəfə
#     artırılır ki, ümumi miqyas saxlansın).
#   - PREDICT zamanı Dropout və BatchNorm "inference" rejimində işləyir:
#     Dropout heç nə söndürmür, BatchNorm öyrətmə zamanı yığılmış hərəkətli
#     statistikadan istifadə edir. Yəni predict nəticəsi təsadüfi sönmədən asılı deyil.
#
# DİQQƏT: funksiya hər çağırışda YENİ qat obyektləri yaradır. Yəni 4 dəfə
# çağırılanda 4 ayrı BatchNorm və 4 ayrı Dropout yaranır, çəkilər paylaşılmır.


# ---------------------------------------------------------------------
# 3) ENCODER-İN QURULMASI (Functional API)
# ---------------------------------------------------------------------
#
#   ENCODER:
#   Giriş      Flatten   Dense+BN+Drop   Dense+BN+Drop
#  (28,28,1) ─► (784,) ─► (256,) ─────► (128,) ──┬──► Dense ──► z_mean    (2,)
#                                               └──► Dense ──► z_log_var (2,)
#                                                         │
#                                      [Sampler] ◄────────┘  → h (2,) (seçilmiş nöqtə)

input_img = Input((28, 28, 1))
# Encoder-in girişi: forma (28, 28, 1). Batch ölçüsü yazılmır.
# (Input(shape=...) əvəzinə qısa yazılış, eyni nəticə.)

x = Flatten()(input_img)
# (28, 28, 1) → (784,). Dense yalnız 1 ölçülü vektorla işləyir. Parametr: yoxdur.
# DİQQƏT: bu x dəyişəni şəbəkənin cari nöqtəsidir, x_train ilə qarışdırmayın.

x = Dense(256, activation='relu')(x)
# 784 → 256, ReLU. Parametr: 784 × 256 + 256 = 200960

x = dropout_and_batch(x)
# BatchNorm (256 neyron üçün 4×256 = 1024 parametr, onlardan 512 öyrənilən)
# + Dropout (parametrsiz).

x = Dense(128, activation='relu')(x)
# 256 → 128, ReLU. Parametr: 256 × 128 + 128 = 32896

x = dropout_and_batch(x)
# BatchNorm (4×128 = 512 parametr, 256 öyrənilən) + Dropout.

z_mean = Dense(hidden_dim)(x)
# Paylanmanın ORTASI (μ). 128 → 2. Parametr: 128 × 2 + 2 = 258.
# Aktivasiya YOXDUR (linear): orta istənilən ədəd ola bilər (mənfi də).
# Mənası: "bu şəkil gizli müstəvidə təxminən (μ1, μ2) nöqtəsindədir".

z_log_var = Dense(hidden_dim)(x)
# Paylanmanın LOQARİFMİK DİSPERSİYASI (log σ²). 128 → 2. Parametr: 258.
# Mənası: "bu mərkəzin ətrafında nə qədər qeyri-müəyyənlik var".
#
# NİYƏ dispersiyanın özü yox, LOQARİFMİ?
#   - Dispersiya həmişə müsbət olmalıdır. Şəbəkənin çıxışı isə istənilən
#     ədəd ola bilər. exp(log_var) isə həmişə müsbətdir → heç bir məhdudiyyət
#     lazım deyil.
#   - Ədədi cəhətdən daha sabitdir (çox kiçik/böyük dispersiyalar üçün).
# Əlaqə: σ² = exp(log_var), yəni σ (standart kənarlaşma) = exp(log_var / 2).


# ---------------------------------------------------------------------
# 4) SƏS-KÜY FUNKSİYASI VƏ SAMPLER QATI
# ---------------------------------------------------------------------

def noiser(args):
    mean, log_var = args
    # batch ölçüsü sabit yazılmır, dinamik götürülür
    N = tf.random.normal(shape=tf.shape(mean), mean=0., stddev=1.0)
    return tf.exp(log_var / 2) * N + mean
# "REPARAMETRİZASİYA HİLLƏSİ" (reparameterization trick):
#
#   z = mean + σ · N,   N ~ Normal(0, 1),   σ = exp(log_var / 2)
#
# Bu, N(mean, σ²) paylanmasından təsadüfi nöqtə seçməyin bir yoludur.
# Birbaşa "paylanmadan seç" əməliyyatı törəmə (gradient) buraxmır, yəni geri
# yayılma (backprop) işləməz. Bu yazılışda təsadüfilik YALNIZ N-dədir (ondan
# öyrənməli çəki yoxdur), mean və log_var isə adi riyazi əməliyyatlarla
# daxil olur → gradient onlardan keçə bilir.
#
# Sətir-sətir:
#   tf.shape(mean)        → mean-in DİNAMİK forması, məsələn (60, 2). Batch ölçüsü
#                           sabit yazılmır; hansı batch gəlsə ona uyğunlaşır
#                           (bu, kodun yarımçıq batch ilə də işləməsini təmin edir).
#   tf.random.normal(...) → həmin formada N(0,1) təsadüfi ədədlər. Hər çağırışda yeni.
#   tf.exp(log_var / 2)   → σ.
#   σ * N + mean          → N(mean, σ²) paylanmasından nümunə.
#
# Mənası: log_var böyük olsa → böyük səs-küy → nöqtə mərkəzdən uzağa düşür;
# log_var çox mənfi olsa → σ ≈ 0 → nöqtə demək olar ki, sabit (adi avtoenkoder kimi).

class Sampler(Layer):
    # Keras 3-də KerasTensor-ları tf funksiyalarına birbaşa vermək olmur,
    # ona görə sampling və KL loss layer daxilində hesablanır
    # Qeyd: Keras 3-də simvolik tensorları (Input-dan gələnləri) çılpaq tf
    # funksiyalarına vermək xəta verir. Ona görə tf əməliyyatları öz qatımızın
    # call metodunda yazılır: qat işlədiyi anda tensorlar artıq real olur.

    def call(self, inputs):
        mean, log_var = inputs
        # inputs = [z_mean, z_log_var] siyahısı, burada açılır.

        kl_loss = -0.5 * tf.reduce_sum(1 + log_var - tf.square(mean) - tf.exp(log_var), axis=-1)
        # KL DİVERGENSİYASI: encoder-in paylanması N(mean, σ²) ilə "ideal"
        # N(0, 1) arasındakı məsafə. Hər şəkil üçün bir ədəd:
        #
        #   KL = -0.5 · Σ ( 1 + log_var − mean² − exp(log_var) )
        #
        # Cəm 2 gizli ölçü üzərindəndir (axis=-1). Forma: (batch,).
        #
        # Bu ifadə nəyə cərimə verir?
        #   - mean²  : mərkəz 0-dan uzaq olsa, cərimə artır → nöqtələr 0 ətrafına çəkilir.
        #   - log_var / exp(log_var): σ² = 1 olanda cərimə minimaldır (0). σ çox
        #     kiçik (qeyri-müəyyənlik yox) və ya çox böyük olsa, cərimə artır.
        # Nəticədə KL bütün şəkillərin paylanmalarını N(0,1) ətrafında toplayır.
        # Bu, gizli fəzanı "kompakt və dəliksiz" edir.

        self.add_loss(tf.reduce_mean(kl_loss))
        # Batch üzrə ortalamanı (bir ədəd) modelin ÜMUMİ itkisinə əlavə edir.
        # Yəni son loss = vae_loss (bərpa xətası) + bu KL.
        # add_loss modelə loss funksiyasından ayrıca bir itki əlavə etməyin yoludur;
        # bu, KL-in encoder daxilində hesablanıb ümumi itkiyə qoşulmasına imkan verir.
        # Encoder, vae-nin içində olsa belə, bu itki avtomatik toplanır.

        return noiser(inputs)
        # Qatın çıxışı: seçilmiş nöqtə z, forma (batch, 2).

    def compute_output_shape(self, input_shape):
        return input_shape[0]
    # Keras-a çıxışın formasını bildirir. input_shape = [mean_forması, log_var_forması],
    # ikisi də (None, 2); çıxış da (None, 2) olduğu üçün ilkini qaytarırıq.

h = Sampler()([z_mean, z_log_var])
# Qatı yaradıb dərhal çağırır. h → (batch, 2), seçilmiş gizli kod z.
# DİQQƏT: h adı sonradan (aşağıda scatter hissəsində) yenidən istifadə olunur
# və orada tamamilə başqa mənası var (encoder.predict nəticəsi). Qarışdırmayın.


# ---------------------------------------------------------------------
# 5) DECODER-İN QURULMASI
# ---------------------------------------------------------------------
#
#   DECODER:
#   Giriş      Dense+BN+Drop   Dense+BN+Drop   Dense       Reshape
#    (2,)  ─► (128,) ─────► (256,) ─────► (784,) ─► (28,28,1)

input_dec = Input(shape=(hidden_dim,))
# Decoder-in ayrıca girişi: 2 ədədlik vektor. Encoder-dən asılı deyil,
# ona görə decoder-ə istənilən 2 ədədi birbaşa verə bilərik (aşağıdakı grid).

d = Dense(128, activation='relu')(input_dec)
# 2 → 128, ReLU. Parametr: 2 × 128 + 128 = 384

d = dropout_and_batch(d)
# BatchNorm (4×128 = 512 parametr) + Dropout.

d = Dense(256, activation='relu')(d)
# 128 → 256, ReLU. Parametr: 128 × 256 + 256 = 33024

d = dropout_and_batch(d)
# BatchNorm (4×256 = 1024 parametr) + Dropout.

d = Dense(28*28, activation='sigmoid')(d)
# 256 → 784, sigmoid (çıxış 0..1, piksel aralığı ilə eyni).
# Parametr: 256 × 784 + 784 = 201488

decoded = Reshape((28, 28, 1))(d)
# (784,) → (28, 28, 1). Parametr: yoxdur.
# QEYD: decoder-də də Dropout var. Bu qeyri-adi seçimdir (adətən Dropout
# decoder-də az işlədilir): öyrətmə zamanı bərpanı bir az "səs-küylü" edir.
# Predict-də söndürülür, ona görə yaradılan şəkillərə təsiri yoxdur.


# ---------------------------------------------------------------------
# 6) MODELLƏRİN YIĞILMASI
# ---------------------------------------------------------------------

encoder = keras.Model(input_img, h, name='encoder')
# Encoder modeli: şəkil girir, SEÇİLMİŞ nöqtə z (2 ədəd) çıxır.
# ÇOX VACİB: çıxış h-dir (Sampler-dən keçmiş), z_mean DEYİL. Yəni
# encoder.predict hər çağırışda eyni şəkil üçün BİR QƏDƏR FƏRQLİ nəticə verir
# (təsadüfi səs-küy var). Mərkəzin özünü (səs-küysüz) almaq istəsəniz:
#   mean_model = keras.Model(input_img, z_mean)
#   mean_model.predict(x_test)
# Parametr sayı (təxmini): 200960 + 1024 + 32896 + 512 + 258 + 258 = 235908.

decoder = keras.Model(input_dec, decoded, name='decoder')
# Decoder modeli: 2 ədəd girir, şəkil çıxır.
# Parametr sayı (təxmini): 384 + 512 + 33024 + 1024 + 201488 = 236432.

vae = keras.Model(input_img, decoder(encoder(input_img)), name="vae")
# Bütöv model: şəkil → encoder (+ Sampler) → z → decoder → şəkil.
# Əvvəlki kodda olduğu kimi, bu üç model ÇƏKİLƏRİ PAYLAŞIR: vae-ni öyrədəndə
# encoder və decoder də öyrənir. Ayrıca fit() lazım deyil.
# KL itkisi Sampler-in içində olduğu üçün vae-nin itkilər siyahısına da düşür.


# ---------------------------------------------------------------------
# 7) İTKİ FUNKSİYASI VƏ KOMPİLYASİYA
# ---------------------------------------------------------------------

def vae_loss(x, y):
    x = tf.reshape(x, (-1, 28*28))
    y = tf.reshape(y, (-1, 28*28))
    loss = tf.reduce_sum(tf.square(x - y), axis=-1)
    return loss  # KL loss Sampler layer-in içində əlavə olunur
# BƏRPA XƏTASI (reconstruction loss). x = orijinal, y = bərpa olunmuş şəkil.
#   tf.reshape(..., (-1, 784)) → (batch, 28, 28, 1) → (batch, 784).
#                                -1 = "batch ölçüsünü özün hesabla".
#   tf.square(x - y)           → hər pikselin kvadratik səhvi.
#   tf.reduce_sum(..., axis=-1)→ 784 pikselin səhvi CƏMLƏNİR (ortalama yox!).
#                                Forma: (batch,), hər şəkil üçün bir ədəd.
#
# NİYƏ CƏM, ORTALAMA YOX? Bu çox vacib məqamdır. Əvvəlki kodda
# 'mean_squared_error' idi (784 pikselin ortalaması, kiçik ədəd ~0.04).
# Burada cəm götürülür (~20-40 civarı). Səbəb: VAE-də ümumi itki =
# bərpa xətası + KL. Əgər bərpa xətası ortalama olsaydı (çox kiçik), KL
# üstün gələrdi, model hər şeyi N(0,1)-ə sıxıb şəkli unudardı (buna "posterior
# collapse" deyilir: bütün şəkillər eyni qeyri-informativ kodla göstərilir,
# decoder isə hamısını ortalama bulanıq lək kimi çəkir). Cəm götürəndə
# iki hissə müqayisə olunan miqyasda olur.
#
# Keras bu (batch,) vektorunu avtomatik batch üzrə ortalayır və üzərinə
# add_loss-dan gələn KL əlavə olunur.
# Loss funksiyasının imzası (y_true, y_pred) olmalıdır; burada x və y adları verilib.

vae.compile(optimizer='adam', loss=vae_loss)
# Optimizer: Adam (defolt öyrənmə sürəti 0.001).
# loss = vae_loss → bərpa xətası; KL isə Sampler-dən avtomatik əlavə olunur.
# YEKUN İTKİ  =  ortalama(bərpa xətası)  +  ortalama(KL).


# ---------------------------------------------------------------------
# 8) ÖYRƏTMƏ
# ---------------------------------------------------------------------

vae.fit(x_train, x_train, epochs=5, batch_size=batch_size, shuffle=True)
# fit(x_train, x_train): giriş də, cavab da eyni şəkildir (avtoenkoder məntiqi).
#
# epochs=5       → bütün 60000 şəkil 5 dəfə keçilir.
# batch_size=60  → hər addımda 60 şəkil. 60000 / 60 = 1000 addım / epoch,
#                  cəmi 5 × 1000 = 5000 çəki yeniləməsi.
# shuffle=True   → hər epoch-da sıra qarışdırılır.
#
# NƏ GÖZLƏMƏK LAZIMDIR?
#   - Loss ədədləri əvvəlki koddan çox BÖYÜKDƏN başlayacaq (cəm + KL). Normaldır.
#   - Epoch sayı azdır (5). Daha çox epoch (məsələn 20-50) şəkilləri bir qədər
#     kəskinləşdirər, lakin 2 ölçülü gizli fəzada bulanıqlıq həmişə qalacaq.
#   - BatchNorm və Dropout fit zamanı "öyrətmə rejimində" işləyir, ona görə
#     ekranda görünən loss predict-dəki real keyfiyyətdən bir qədər pis ola bilər.


# ---------------------------------------------------------------------
# 9) GİZLİ FƏZANIN GÖRÜNTÜSÜ
# ---------------------------------------------------------------------

h = encoder.predict(x_test[:6000], batch_size=batch_size)
# İlk 6000 test şəklini encoder-dən keçirir. h-in forması: (6000, 2).
# (Burada h Sampler-in simvolik çıxışı yox, real ədədlər massividir; ad
# təkrar istifadə olunub, model artıq qurulub, ona görə problem yoxdur.)
# Hər şəkil müstəvidə bir nöqtədir. DİQQƏT: h seçilmiş (səs-küylü) kodlardır,
# təmiz mərkəzlər deyil, ona görə bulud bir az "dağınıq" görünür.
# Yalnız 6000 şəkil götürülüb: sürət üçün və qrafik daha oxunaqlı olsun deyə.

plt.scatter(h[:, 0], h[:, 1], s=2)
plt.show()
# Nöqtə buludu. s=2 → kiçik nöqtələr (çox nöqtə olduqda üst-üstə düşməsin).
# plt.show() burada VAR → qrafik bitirilir, sonrakı qrafik ayrı figurda çəkilir
# (əvvəlki koddakı üst-üstə düşmə problemi yoxdur).
#
# NƏYƏ BAXMALI?
#   - Bulud təxminən 0 ətrafında yığılıb, miqyas ~(-3..3). Əvvəlki adi
#     avtoenkoderdə miqyas 0..300 ola bilərdi. Fərq KL itkisindəndir.
#   - Bulud təxminən dairəvi və fasiləsizdir: nöqtələr arasında böyük boşluq yoxdur.
#   - Rənglənməyib; sinifləri görmək üçün:
#       plt.scatter(h[:, 0], h[:, 1], s=2, c=y_test[:6000], cmap='tab10')
#       plt.colorbar()
#     (y_test[:6000] — x_test[:6000] ilə eyni sıra.)


# ---------------------------------------------------------------------
# 10) GİZLİ FƏZANIN XƏRİTƏSİ — DECODER İLƏ GRİD
# ---------------------------------------------------------------------

n = 5
# Hər istiqamətdə mərkəzdən neçə addım atılacağı.

total = 2*n+1
# Grid-in ölçüsü: 2·5 + 1 = 11 → 11 × 11 = 121 şəkil.
# (−5, −4, ..., 0, ..., 4, 5 → 11 dəyər.)

plt.figure(figsize=(total/2, total/2))
# Figur ölçüsü 11/2 × 11/2 düym (hər xana təxminən 1/2 düym).

num = 1
# subplot nömrəsi (1-dən başlayır). Soldan sağa, yuxarıdan aşağı sayılır.

for i in range(-n, n+1):
    for j in range(-n, n+1):
        # i və j hər biri -5..5. i → sətir (1-ci gizli ölçü), j → sütun (2-ci gizli ölçü).

        ax = plt.subplot(total, total, num)
        num += 1
        # 11×11 şəbəkənin num-cu xanası.

        img = decoder.predict(np.expand_dims([3*i/n, 3*j/n], axis=0), verbose=0)
        # Gizli fəzada nöqtə: (3·i/5, 3·j/5). i = -5..5 → -3.0, -2.4, -1.8, -1.2,
        # -0.6, 0, 0.6, ..., 3.0 (addım 0.6).
        # NİYƏ -3..3? KL itkisi bütün kodları N(0,1) ətrafına toplayır. Normal
        # paylanmada nöqtələrin ~99.7%-i 3 σ daxilindədir. Yəni -3..3 kvadratı
        # decoder-in öyrəndiyi bölgəni əhatə edir. Əvvəlki kodda bu aralığı
        # əl ilə seçmək lazım idi və riskli idi; burada ardıcıldır.
        # np.expand_dims(..., axis=0): (2,) → (1, 2) (batch ölçüsü).
        # verbose=0 → hər predict çağırışında irəliləyiş cədvəli çıxmasın
        # (yoxsa 121 dəfə çap olunardı).
        # Qeyd: 121 ayrı predict çağırışı yavaşdır. Sürətli variant: bütün 121
        # nöqtəni bir massivdə yığıb decoder.predict-i bir dəfə çağırmaq.

        plt.imshow(img.squeeze(), cmap='gray')
        # (1, 28, 28, 1) → (28, 28), boz tonlu şəkil.

        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        # Oxları, rəqəmləri gizlədir ki, şəkillər təmiz bitişik görünsün.

plt.show()
# 11×11 şəkil xəritəsini göstərir.
#
# NƏYƏ BAXMALI?
#   - Rəqəmlər bir-birinə hamar şəkildə keçir (məsələn, 1 → 7 → 9 → 4).
#     Bu, VAE-nin əsas üstünlüyüdür: gizli fəza fasiləsizdir.
#   - Xəritənin mərkəzi (0, 0) ətrafında rəqəmlər daha "qarışıq" və bulanıq,
#     kənarlarda isə daha aydın rəqəmlər görünə bilər.
#   - 2 ölçüdə keyfiyyət məhduddur: bəzi rəqəmlər (məsələn 4/9, 3/5/8)
#     bir-birinə çox keçir. Daha yaxşı keyfiyyət üçün hidden_dim artırılmalı
#     (amma onda 2D xəritə qurmaq mümkün olmaz).